# TD 3 · Entraîner en pratique : le pipeline PyTorch sur données réelles

**Bloc 3 : deep learning · Séance 9**

---

## Objectifs

À la fin de ce TD, vous serez capable de :

1. manipuler des tenseurs (types, formes) et obtenir un gradient avec autograd ;
2. préparer des données tabulaires pour un réseau **sans data leakage**, et expliquer
   par le calcul pourquoi le feature scaling est indispensable ;
3. écrire un `Dataset`, utiliser un `DataLoader`, et relier batch, itération et epoch ;
4. écrire la training loop complète et suivre les courbes dès l'epoch 1 ;
5. comparer SGD, momentum et Adam, et tester un learning rate schedule ;
6. mesurer l'effet de l'initialisation des poids.

## Prérequis

- TD 1 et TD 2 (la boucle à la main) ;
- séance 3 (pipeline et data leakage), séance 6 (GroupKFold par hôte) ;
- cours : partie 2 du diaporama (séance 9).

## Concepts mobilisés

tenseur · `dtype` · broadcasting · autograd · `ColumnTransformer` ajusté dans le fold ·
feature scaling · `Dataset`, `DataLoader` · batch, itération, epoch ·
`model.train()`, `model.eval()`, `torch.no_grad()` · SGD, momentum, Adam ·
learning rate schedule · initialisation (He).

## Déroulé

| Exercice | Contenu | Durée | Type | Niveau |
|---|---|---|---|---|
| 3.1 | Tenseurs et autograd | 10 min | guidé | ★☆☆ |
| 3.2 | Préparer les données : le fold et le feature scaling | 15 min | compléter, analyser | ★★☆ |
| 3.3 | `Dataset` et `DataLoader` | 10 min | compléter | ★☆☆ |
| 3.4 | La training loop | 15 min | compléter, vérifier | ★★☆ |
| 3.5 | Optimizers : SGD, momentum, Adam | 15 min | expérimenter, comparer | ★★☆ |
| 3.6 | Planifier le learning rate | 10 min | expérimenter | ★★☆ |
| 3.7 | L'initialisation | 10 min | comparer | ★★☆ |
| | Synthèse et questions | 5 min | | |

## Résultats attendus

- Un gradient autograd identique à la formule écrite à la main.
- 14 colonnes d'origine devenues 77 features (11 numériques, 66 indicatrices) ; des
  gradients qui diffèrent de plusieurs ordres de grandeur entre features sans scaling.
- 40 mises à jour par epoch avec des batches de 256.
- Votre boucle retrouve **exactement** la courbe de la séance 9 : minimum de
  validation 0,4424 à l'epoch 8.
- SGD à 0,01 rampe ; le momentum 0,9 au même learning rate rattrape SGD à 0,1 ; Adam
  descend le plus vite.
- Une initialisation nulle ramène l'AUC à 0,5 ; une initialisation trop large fait
  exploser la loss de la première epoch.

## Mode d'emploi

Les expériences tournent sur **un seul fold** (le fold 0) et une seule seed : elles
illustrent des tendances, elles ne prouvent rien à elles seules (séance 6). Traitez
toute différence d'AUC inférieure à 0,01 comme du bruit.

## Mise en route

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import math
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.data import SEED, load_raw, set_seed, split_by_host
from src.training import (EarlyStopping, TabularDataset, evaluate, fit,
                          loaders_from_arrays, make_mlp)

set_seed(SEED)
torch.manual_seed(SEED)
# Quatre threads, pour deux raisons. (1) Sur des réseaux aussi petits, davantage de
# threads RALENTIT l'entraînement : le coût de coordination dépasse le gain. (2) Le
# nombre de threads change l'ordre des additions en virgule flottante, donc les
# derniers chiffres des résultats : les valeurs de référence des séances 9 et 10 ont
# été obtenues avec 4 threads. Même seed, autre nombre de threads : autres décimales.
torch.set_num_threads(4)

# Les mêmes données, la même cible et les mêmes folds qu'en séances 7 et 9.
df = load_raw()
train, _ = split_by_host(df)
train["licensed"] = train["license"].map(
    lambda s: bool(re.search(r"HUT|HB-|AJ0|ESFC", str(s))) if str(s) != "nan" else False)
NUMERIC = ["accommodates", "bedrooms", "beds", "bathrooms", "latitude", "longitude",
           "minimum_nights", "number_of_reviews", "review_scores_rating",
           "calculated_host_listings_count", "availability_365"]
CATEGORICAL = ["room_type", "property_type", "neighbourhood_cleansed"]
X = train[NUMERIC + CATEGORICAL]
y = train["licensed"].astype(int)
FOLDS = list(GroupKFold(n_splits=5).split(X, y, groups=train["host_id"]))
train_idx, val_idx = FOLDS[0]
print(f"annonces {len(train):,}   part de licences {y.mean():.4f}   torch {torch.__version__}")

---

## Exercice 3.1 · Tenseurs et autograd

*Guidé · 10 min · ★☆☆*

**Explication.** Un **tenseur** est un tableau à la NumPy, avec deux capacités de
plus : il peut vivre sur un GPU, et il peut **enregistrer** les opérations qu'il
subit pour calculer des gradients (`requires_grad=True`). Deux détails piègent tout le
monde : le type (`float32` par défaut pour PyTorch, `float64` pour NumPy) et la forme.

**Exemple.**

In [ ]:
a = torch.tensor([1.0, 2.0])                       # créé par PyTorch : float32
b = torch.tensor(np.array([1.0, 2.0]))             # venu de NumPy : garde float64
print(f"  types : {a.dtype} contre {b.dtype}")

prediction = torch.zeros(4, 1)                     # sortie d'un réseau : (n, 1)
cible = torch.tensor([1.0, 0.0, 1.0, 0.0])         # une cible mal formée : (n,)
print(f"  (4, 1) - (4,) donne une forme {tuple((prediction - cible).shape)} : "
      "le broadcasting a fabriqué une matrice")
print(f"  avec la cible en colonne : {tuple((prediction - cible.reshape(-1, 1)).shape)}")

**À vous.** Obtenez un gradient avec autograd et comparez-le à la formule.

In [ ]:
# TODO : pour L(w, b) = moyenne de (w*x + b - y)^2, avec x = [1, 2, 3, 4],
# y = [2, 4, 6, 8], w = 0,5 et b = 0 :
#  (1) créez w et b avec requires_grad=True, calculez L, appelez L.backward() ;
#  (2) comparez w.grad et b.grad aux formules dL/dw = moyenne(2 (w x + b - y) x) et
#      dL/db = moyenne(2 (w x + b - y)).
grad_autograd, grad_formule = None, None

In [ ]:
if grad_autograd is None:
    print("  à compléter")
else:
    ok = np.allclose(grad_autograd, grad_formule, atol=1e-6)
    print(f"  {'OK' if ok else 'à revoir'}")

**Analyse.** autograd a appliqué la chain rule que vous avez écrite au TD 2, sur
une fonction que vous n'avez jamais dérivée pour lui.

**Question 3.1.** Pourquoi une sortie de forme (4, 1) comparée à une cible de forme
(4,) est-elle dangereuse, alors qu'aucune erreur n'est levée ?

---

## Exercice 3.2 · Préparer les données : le fold et le feature scaling

*Compléter, analyser · 15 min · ★★☆*

**Explication.** Deux règles de la séance 3 reviennent, sans `Pipeline` pour les
imposer :

1. le préprocesseur s'ajuste (`fit`) sur la partie entraînement du fold **seulement** ;
2. un réseau a besoin d'un **feature scaling** de ses entrées, et la raison est
   mécanique : le gradient d'un poids est proportionnel à la valeur de l'entrée qu'il
   multiplie.

**Exemple.** Les étendues brutes des features numériques (partie entraînement du fold).

In [ ]:
brut = X.iloc[train_idx][NUMERIC]
print(brut.describe().loc[["min", "max"]].T.round(2).to_string())

In [ ]:
# TODO : construisez le préprocesseur de la séance 9 (médiane + StandardScaler pour
# NUMERIC ; mode + OneHotEncoder(handle_unknown="ignore", min_frequency=20,
# sparse_output=False) pour CATEGORICAL), ajustez-le sur X.iloc[train_idx] SEULEMENT,
# puis transformez les deux parties du fold.
# Affichez les formes, et le nombre de colonnes indicatrices.
prep = None
X_tr = X_va = None
y_tr = y.iloc[train_idx].to_numpy()
y_va = y.iloc[val_idx].to_numpy()

**À vous : analyser le mécanisme.** Prenez un seul neurone sigmoïde dont tous les
poids valent zéro : il prédit 0,5 pour tout le monde. Deux quantités décrivent alors
la surface de la loss le long du poids wⱼ :

- le **gradient**, moyenne((0,5 - y)·xⱼ), proportionnel aux valeurs de xⱼ ;
- la **courbure**, 0,25·moyenne(xⱼ²), qui fixe le plus grand learning rate stable dans
  cette direction : η < 2 / courbure. Au-delà, le gradient descent oscille et diverge
  le long de wⱼ.

Comparez ces quantités avant et après feature scaling.

In [ ]:
# TODO : pour chaque feature de NUMERIC, calculez le gradient, la courbure et le learning
# rate stable maximal 2 / courbure :
#   (a) sur les données imputées mais SANS scaling
#       (SimpleImputer(strategy="median") ajusté sur l'entraînement) ;
#   (b) sur les 11 premières colonnes de X_tr (standardisées).
# Affichez un tableau, puis le rapport entre le plus grand et le plus petit learning rate
# stable dans chaque cas.

![Gradient descent sur une cuvette étirée ou ronde](../assets/generated/cours_08_10_fr/f07_descente_echelle.png)

**Analyse.** Sans feature scaling, le learning rate stable varie de plusieurs **ordres
de grandeur** d'une feature à l'autre : un learning rate assez petit pour
`availability_365` (valeurs jusqu'à 365) fait ramper les poids des features proches
de 1, et un learning rate adapté à celles-ci fait diverger les autres. La surface de
la loss est une cuvette étirée, et le gradient descent zigzague (figure). Remarquez
aussi le gradient brut de `latitude` : il est grand parce que ses valeurs tournent
autour de 41, pas parce qu'elle serait informative ; ses variations utiles (un
dixième de degré) sont noyées. Une fois les features standardisées, la courbure vaut
0,25 partout et le learning rate stable est le même pour toutes : c'est une
conséquence du calcul, pas une règle d'hygiène.

**Question 3.2.** Un decision tree aurait-il besoin de ce scaling ?
Pourquoi ?

---

## Exercice 3.3 · `Dataset` et `DataLoader`

*Compléter · 10 min · ★☆☆*

**Explication.** Un `Dataset` dit comment obtenir **un** exemple ; un `DataLoader` en
fait des batches, les mélange à chaque epoch et les parcourt. Trois mots à ne pas
confondre :

| terme | sens |
|---|---|
| **batch** | les exemples utilisés pour un gradient step |
| **itération** | un gradient step |
| **epoch** | un passage complet sur le training set |

In [ ]:
# TODO : complétez MonDataset. Les X en float32 ; la cible en float32 et en COLONNE.


class MonDataset(Dataset):
    def __init__(self, X, y):
        # TODO
        raise NotImplementedError

    def __len__(self):
        # TODO
        raise NotImplementedError

    def __getitem__(self, i):
        # TODO : renvoyer (x_i, y_i)
        raise NotImplementedError

In [ ]:
# Vérification contre le TabularDataset du module src/training.py.
mien, reference = MonDataset(X_tr, y_tr), TabularDataset(X_tr, y_tr)
ok = (len(mien) == len(reference) and torch.equal(mien[0][0], reference[0][0])
      and torch.equal(mien[0][1], reference[0][1]))
print(f"  {len(mien)} exemples, premier exemple identique : {'OK' if ok else 'à revoir'}")

In [ ]:
# TODO : pour des batches de 64, 256 et 1024 (DataLoader(..., shuffle=True)), affichez
# le nombre de batches par epoch (len(loader)), la forme du premier batch, et le nombre
# de mises à jour pour 20 epochs.

**Analyse.** À nombre d'epochs égal, des batches de 1024 font **seize fois moins** de
mises à jour que des batches de 64. Comparer deux batch sizes « à 20 epochs » compare
donc aussi deux nombres de steps : ce sera la clé du résultat de la séance 9 (la
batch size achète de la vitesse, pas de la qualité).

**Question 3.3.** Pourquoi mélanger (`shuffle=True`) à chaque epoch plutôt qu'une fois
pour toutes ?

---

## Exercice 3.4 · La training loop

*Compléter, vérifier · 15 min · ★★☆*

**Explication.** La training loop de la séance 8, avec trois ajouts pour les vraies
données :

- `model.train()` avant l'entraînement et `model.eval()` avant l'évaluation (sans
  effet aujourd'hui, **indispensable** dès qu'il y aura du dropout, TD 5) ;
- `torch.no_grad()` pendant l'évaluation : pas de computational graph, moins de
  mémoire ;
- la loss de chaque epoch pondérée par la taille de chaque batch (le dernier batch est
  plus court).

In [ ]:
# TODO : écrivez la training loop, 25 epochs, et remplissez historique["train"] et
# historique["val"] à chaque epoch.
torch.manual_seed(SEED)
modele = make_mlp(X_tr.shape[1], hidden=(64, 32))
perte_fn = nn.BCEWithLogitsLoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=1e-3)
chargeur_tr, chargeur_va = loaders_from_arrays(X_tr, y_tr, X_va, y_va,
                                               batch_size=256, seed=SEED)
historique = {"train": [], "val": []}
# for epoque in range(25):
#     modele.train()
#     ...  (zero_grad, forward pass, loss, backward, step, cumul pondéré par len(xb))
#     modele.eval()
#     with torch.no_grad():
#         ...

In [ ]:
# Vérification : la séance 9 avait trouvé un minimum de validation de 0,4424 à
# l'epoch 8, avec exactement ces réglages.
if not historique["val"]:
    print("  à compléter")
else:
    meilleure = int(np.argmin(historique["val"]))
    print(f"  epoch 1 : entraînement {historique['train'][0]:.4f}  validation {historique['val'][0]:.4f}")
    print(f"  minimum de validation {historique['val'][meilleure]:.4f} à l'epoch {meilleure + 1}")
    ok = abs(historique["val"][meilleure] - 0.4424) < 1e-3 and meilleure + 1 == 8
    print(f"  identique à la séance 9 : {'OK' if ok else 'à revoir'}")
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.plot(range(1, 26), historique["train"], label="entraînement")
    ax.plot(range(1, 26), historique["val"], label="validation")
    ax.axvline(meilleure + 1, color="grey", ls=":", label=f"minimum ({meilleure + 1})")
    ax.set(xlabel="epoch", ylabel="cross-entropy", title="Suivre la validation dès l'epoch 1")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Votre boucle contre fit() de src/training.py : même seed, mêmes chargeurs.
torch.manual_seed(SEED)
modele_fit = make_mlp(X_tr.shape[1], hidden=(64, 32))
tr_f, va_f = loaders_from_arrays(X_tr, y_tr, X_va, y_va, batch_size=256, seed=SEED)
historique_fit = fit(modele_fit, tr_f, va_f, epochs=25, lr=1e-3, seed=SEED)
if historique["val"]:
    ecart = np.abs(np.array(historique["val"]) - np.array(historique_fit["val"])).max()
    print(f"  écart maximal entre votre boucle et fit() : {ecart:.1e}")

**Analyse.** Le minimum de validation arrive à l'epoch 8 ; ensuite, la training loss
continue de baisser pendant que la validation loss remonte : **dix-sept epochs de
mémorisation**. `fit()` de `src/training.py` n'est rien d'autre que votre training
loop, rangée dans une fonction : les deux coïncident.

**Question 3.4.** Que se serait-il passé si vous n'aviez affiché que la loss finale ?

---

## Exercice 3.5 · Optimizers : SGD, momentum, Adam

*Expérimenter, comparer · 15 min · ★★☆*

**Explication.**

| optimizer | idée | mise à jour |
|---|---|---|
| SGD | un step contre le gradient du batch | θ ← θ - η·g |
| SGD + momentum | une vitesse qui accumule les gradients passés | v ← β·v + g ; θ ← θ - η·v |
| Adam | moyennes mobiles du gradient et de son carré : un step adapté à chaque paramètre | θ ← θ - η·m̂ / (√v̂ + ε) |

![Trois optimizers sur un ravin](../assets/generated/cours_08_10_fr/f08_optimiseurs.png)

In [ ]:
# TODO : pour chacune des configurations ci-dessous, entraînez make_mlp(77, (64, 32))
# pendant 15 epochs (batches de 256, seed SEED, fit() avec optimizer=...), puis relevez
# la training loss aux epochs 1, 5 et 15, le minimum de validation et l'AUC.
#   "SGD learning rate 0,01"     : torch.optim.SGD(params, lr=0.01)
#   "SGD learning rate 0,1"      : torch.optim.SGD(params, lr=0.1)
#   "SGD + momentum 0,9, 0,01"   : torch.optim.SGD(params, lr=0.01, momentum=0.9)
#   "Adam learning rate 0,001"   : torch.optim.Adam(params, lr=1e-3)

**Analyse.** SGD au learning rate 0,01 **rampe** : après 15 epochs, il n'a pas fini
de descendre, et son AUC le montre. Le momentum 0,9 **au même learning rate** se
comporte presque comme SGD au learning rate 0,1. Adam fait descendre la training loss
le plus vite, sans réglage : c'est pour cela qu'il est le choix par défaut. Mais une
training loss plus basse n'est pas une meilleure validation : comparez la colonne
« val min ».

**Question 3.5.** Pourquoi le momentum 0,9 au learning rate 0,01 ressemble-t-il à SGD
au learning rate 0,1 ? (Indice : que vaut la vitesse v quand le gradient reste à peu
près constant ?)

---

## Exercice 3.6 · Planifier le learning rate

*Expérimenter · 10 min · ★★☆*

**Explication.** Un learning rate constant est un compromis : assez grand pour avancer
au début, trop grand pour se poser à la fin. Un **learning rate schedule** fait évoluer
le learning rate : par paliers, en cosinus, avec un **warm-up** au départ.

![Quatre learning rate schedules](../assets/generated/cours_08_10_fr/f09_planification.png)

In [ ]:
# TODO : écrivez entrainer_planifie(fabrique_planif) qui entraîne make_mlp(77, (64, 32))
# 20 epochs avec SGD(lr=0.05, momentum=0.9), batches de 256, et appelle planif.step()
# à la fin de chaque epoch. Comparez :
#   "constant"    : aucun schedule ;
#   "paliers"     : torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.1) ;
#   "cosinus"     : torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20).
# Relevez la validation loss finale, son minimum, et l'AUC finale.

**Analyse.** Avec un learning rate constant, la validation loss finale s'éloigne de
son minimum : les derniers steps, trop grands, continuent d'agiter le modèle. En
réduisant le learning rate en fin de parcours, les schedules terminent plus près du
minimum. L'écart d'AUC est petit, sur un seul fold et une seule seed : c'est une
**tendance à confirmer**, pas un résultat.

**Question 3.6.** À quoi sert un warm-up (learning rate faible pendant les premières
epochs) ? Reliez votre réponse aux dead units (dying ReLU) du TD 2.

*Pour aller plus loin (ouvert) :* relancez les trois schedules avec
`graine=1` et `graine=2`. L'ordre des AUC tient-il ?

---

## Exercice 3.7 · L'initialisation

*Comparer · 10 min · ★★☆*

**Explication.** Au TD 2, une initialisation nulle bloquait tout. Sur un vrai réseau,
l'échelle compte aussi : trop petite, le signal s'éteint de couche en couche ; trop
grande, il explose. L'initialisation de **He** (He et al., 2015) tire les poids avec
une variance 2 / (nombre d'entrées), pensée pour ReLU, qui annule la moitié des
valeurs.

In [ ]:
# TODO : écrivez initialiser(net, mode) qui parcourt les nn.Linear du réseau et applique :
#   "défaut PyTorch" : rien ;
#   "zéros"          : nn.init.zeros_ sur poids et biais ;
#   "constante 0,1"  : nn.init.constant_(poids, 0.1), biais à zéro ;
#   "normale σ = 3"  : nn.init.normal_(poids, 0, 3.0) ;
#   "He"             : nn.init.kaiming_normal_(poids, nonlinearity="relu"), biais à zéro.
# Puis entraînez chaque variante 15 epochs (batches de 512, Adam 1e-3) et relevez l'AUC,
# la training loss de l'epoch 1 et le nombre d'unités distinctes de la première
# couche : len(np.unique(net[0].weight.detach().numpy().round(6), axis=0)).

**Analyse.** Avec des zéros, rien n'apprend (AUC 0,5) ; avec une constante, les 64
unités restent identiques : il n'y a jamais de symmetry breaking, comme au TD 2. Avec
des poids trop grands, la loss de la première epoch explose et le réseau ne s'en
remet pas en 15 epochs. L'initialisation par défaut de PyTorch et celle de He
donnent des résultats proches : sur un fold et une seed, leur écart ne permet pas de
les départager.

**Question 3.7.** Pourquoi la variance de He contient-elle un facteur 2 que n'a pas
celle de Glorot (Xavier), pensée pour tanh ?

---

## Synthèse

- Un tenseur a un **type** et une **forme** ; le broadcasting de (n, 1) contre (n,)
  fabrique silencieusement une matrice (n, n).
- Le préprocesseur s'ajuste **dans le fold** ; le feature scaling est indispensable
  parce que le gradient est proportionnel à l'entrée.
- Une epoch = n / batch size itérations : 40 pour des batches de 256 ici.
- La training loop : `train()`, `zero_grad`, forward pass, loss, `backward`, `step` ;
  puis `eval()` et `no_grad()` pour la validation. **Suivre la validation dès l'epoch 1.**
- Le momentum multiplie le step effectif par 1/(1 - β) et amortit les oscillations ;
  Adam adapte le step à chaque paramètre.
- Réduire le learning rate en fin d'entraînement stabilise la fin de courbe.
- Une initialisation nulle ou constante empêche le symmetry breaking ; une
  initialisation trop large fait exploser la loss.

## Questions de compréhension

1. Donnez le mécanisme (pas la règle) qui rend le feature scaling nécessaire.
2. Combien de mises à jour fait une epoch avec des batches de 64 sur 10 100 lignes ?
3. Pourquoi faut-il `torch.no_grad()` pendant l'évaluation ?
4. Que fait le momentum quand le gradient change de signe d'une itération à l'autre ?
5. Pourquoi une initialisation constante empêche-t-elle d'apprendre ?

## Pour aller plus loin

- PyTorch, *Datasets & DataLoaders*.
  https://pytorch.org/tutorials/beginner/basics/data_tutorial.html
- Goh, G. (2017), *Why Momentum Really Works*, Distill (licence CC BY 4.0, figures
  interactives). https://distill.pub/2017/momentum/
- Kingma, D. et Ba, J. (2015), *Adam: A Method for Stochastic Optimization*, ICLR.
  https://arxiv.org/abs/1412.6980
- Loshchilov, I. et Hutter, F. (2017), *SGDR: Stochastic Gradient Descent with Warm
  Restarts*, ICLR (learning rate schedule en cosinus). https://arxiv.org/abs/1608.03983
- Goyal, P. et al. (2017), *Accurate, Large Minibatch SGD: Training ImageNet in 1
  Hour* (warm-up). https://arxiv.org/abs/1706.02677
- Glorot, X. et Bengio, Y. (2010), *Understanding the difficulty of training deep
  feedforward neural networks*, AISTATS.
- He, K. et al. (2015), *Delving Deep into Rectifiers*.
  https://arxiv.org/abs/1502.01852

**Prochain TD :** diagnostiquer un entraînement, dimensionner un réseau, et le
comparer honnêtement au champion de la séance 7.